# DefaultWorkflow

This notebook was generated from the Curio workflow `Example_1_step5.json`.

It computes the UTCI (Universal Thermal Climate Index) from raster and weather data, then performs zonal statistics over socio-demographic regions in Milan.

## Step 1 — Load Raster Data (Mean Radiant Temperature)

In [2]:
import rasterio

timestamp = 12
src = rasterio.open(f'milan/Milan_Tmrt_2022_203_{timestamp:02d}00D.tif')

src

<open DatasetReader name='milan/Milan_Tmrt_2022_203_1200D.tif' mode='r'>

## Step 2 — Load Weather Sensor Data

In [3]:
import pandas as pd

sensor = pd.read_csv('milan/Milan_22.07.2022_Weather_File_UMEP_CSV.csv', delimiter=';')

sensor

,%iy,id,it,imin,Q*,QH,QE,Qs,Qf,Wind,...,Kdn,snow,ldown,fcld,wuh,xsmd,lai_hr,Kdiff,Kdir,Wd
0,2022,203,0,0,-999,-999,-999,-999,-999,0.7,...,0,0,398,0.60,-999,-999,-999,0,0,142
1,2022,203,1,0,-999,-999,-999,-999,-999,1.0,...,0,0,401,0.67,-999,-999,-999,0,0,198
2,2022,203,2,0,-999,-999,-999,-999,-999,0.8,...,0,0,404,0.71,-999,-999,-999,0,0,320
3,2022,203,3,0,-999,-999,-999,-999,-999,1.2,...,0,0,397,0.15,-999,-999,-999,0,0,7
4,2022,203,4,0,-999,-999,-999,-999,-999,1.6,...,0,0,391,0.14,-999,-999,-999,0,0,33
5,2022,203,5,0,-999,-999,-999,-999,-999,1.9,...,0,0,388,0.07,-999,-999,-999,0,0,40
6,2022,203,6,0,-999,-999,-999,-999,-999,1.2,...,38,0,386,0.12,-999,-999,-999,20,18,36
7,2022,203,7,0,-999,-999,-999,-999,-999,1.2,...,166,0,385,0.13,-999,-999,-999,58,108,66
8,2022,203,8,0,-999,-999,-999,-999,-999,1.5,...,326,0,401,0.06,-999,-999,-999,85,241,106
9,2022,203,9,0,-999,-999,-999,-999,-999,1.6,...,453,0,410,0.07,-999,-999,-999,100,353,122


## Step 3 — Load Socio-Demographic Shapefile

In [3]:
import geopandas as gpd

gdf = gpd.read_file('milan/R03_21-11_WGS84_P_SocioDemographics_MILANO_Selected.shp')

gdf

,COD_REG,COD_ISTAT,PRO_COM,SEZ2011,SEZ,LOC2011,COD_LOC,TIPO_LOC,COM_ASC,COD_ASC,...,PF7,PF8,TotPOP,lt_5,gt_65,Unemployed,UE_Foreig,noUE_Forei,HousH_gt_5,geometry
0,3.0,3015146.0,15146.0,151460005984,5984.0,1.514610e+09,10006.0,1.0,15146009.0,9,...,6,4,185,8,49,43,0,20,10,"POLYGON ((516035.556 5040991.638, 516069.285 5..."
1,3.0,3015146.0,15146.0,151460001209,1209.0,1.514610e+09,10006.0,1.0,15146002.0,2,...,2,0,122,7,24,23,1,22,2,"POLYGON ((517516.125 5040315.897, 517532.085 5..."
2,3.0,3015146.0,15146.0,151460001062,1062.0,1.514610e+09,10006.0,1.0,15146002.0,2,...,5,2,455,14,79,89,6,148,7,"POLYGON ((517570.432 5038480.323, 517515.858 5..."
3,3.0,3015146.0,15146.0,151460005439,5439.0,1.514610e+09,10006.0,1.0,15146009.0,9,...,6,0,310,5,73,61,3,41,6,"POLYGON ((514837.673 5038687.087, 514898.502 5..."
4,3.0,3015146.0,15146.0,151460000748,748.0,1.514610e+09,10006.0,1.0,15146002.0,2,...,4,0,222,4,53,43,1,57,4,"POLYGON ((515320.715 5038339.251, 515327.355 5..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6080,3.0,3015146.0,15146.0,151460002316,2316.0,1.514610e+09,10006.0,1.0,15146004.0,4,...,0,0,14,0,3,2,0,4,0,"POLYGON ((520945.266 5035272.493, 520829.733 5..."
6081,3.0,3015146.0,15146.0,151460003011,3011.0,1.514627e+09,26602.0,2.0,15146005.0,5,...,0,0,2,0,0,1,0,0,0,"POLYGON ((515132.585 5029647.805, 515151.6 502..."
6082,3.0,3015146.0,15146.0,151460003017,3017.0,1.514627e+09,26602.0,2.0,15146005.0,5,...,1,0,7,1,0,2,0,1,1,"POLYGON ((515103.596 5029458.179, 515205.059 5..."
6083,3.0,3015146.0,15146.0,151460003028,3028.0,1.514627e+09,26704.0,2.0,15146005.0,5,...,0,0,2,0,0,1,0,1,0,"POLYGON ((514410.811 5029162.241, 514667.955 5..."


## Step 4 — Compute UTCI (Universal Thermal Climate Index)

In [4]:
import xarray as xr
from pythermalcomfort import models
import numpy as np
from rasterio.warp import Resampling

upscale_factor = 0.25
dataset = src
data = dataset.read(
    out_shape=(
        dataset.count,
        int(dataset.height * upscale_factor),
        int(dataset.width * upscale_factor)
    ),
    resampling=Resampling.nearest,
    masked=True
)
data.data[data.data == src.nodatavals[0]] = np.nan

sensor_filtered = sensor[sensor['it'] == timestamp]
tdb = sensor_filtered['Td'].values[0]
v = sensor_filtered['Wind'].values[0]
rh = sensor_filtered['RH'].values[0]


def xutci(tdb, tr, v, rh, units='SI'):
    # Use vectorize=True to properly handle array operations
    result = xr.apply_ufunc(
        models.utci,
        tdb,
        tr,
        v,
        rh,
        units,
        vectorize=True  # This ensures proper handling of the function
    )
    # If the result is still a UTCI object, extract the value
    if hasattr(result, 'values'):
        return result.values
    return result


utci = xutci(tdb, data[0], v, rh)

# Convert to list - handle both numpy array and xarray cases
if isinstance(utci, np.ndarray):
    utci_list = utci.tolist()
elif hasattr(utci, 'tolist'):
    utci_list = utci.tolist()
else:
    # If it's still a UTCI object, try to get the underlying data
    utci_list = np.array(utci).tolist()

utci_shape = [data.shape[-1], data.shape[-2]]

print(f'UTCI grid shape: {utci_shape}')
utci_list

UTCI grid shape: [4632, 4142]


------------------------------------------------------------------------------
                                     UTCI                                     
------------------------------------------------------------------------------
utci : [[nan, nan, nan, nan, nan, ..., nan, nan, nan, nan, nan], [nan, nan,...
stress_category : [[nan, nan, nan, nan, nan, ..., nan, nan, nan, nan, nan],...

## Step 5 — Zonal Statistics (UTCI per Socio-Demographic Region)

In [8]:
# Step 5 — Zonal Statistics (replace entire cell)

from rasterstats import zonal_stats
import numpy as np

utci = np.asarray(utci_list, dtype=float)
shape = utci_shape

if utci.ndim != 2:
    raise ValueError(f"Expected 2D UTCI array, got shape={utci.shape}, ndim={utci.ndim}")

transform = dataset.transform * dataset.transform.scale(
    (dataset.width / shape[0]),
    (dataset.height / shape[1]),
)

# Avoid nodata warning and make nodata explicit
nodata_value = -999.0
utci_for_stats = np.where(np.isnan(utci), nodata_value, utci)

joined = zonal_stats(
    gdf,
    utci_for_stats,
    stats=["min", "max", "mean", "median"],
    affine=transform,
    nodata=nodata_value,
)

gdf["mean"] = [d["mean"] for d in joined]
result = gdf.loc[:, [gdf.geometry.name, "mean", "gt_65"]]
result

,geometry,mean,gt_65
0,"POLYGON ((516035.556 5040991.638, 516069.285 5...",39.557638,49
1,"POLYGON ((517516.125 5040315.897, 517532.085 5...",39.839299,24
2,"POLYGON ((517570.432 5038480.323, 517515.858 5...",40.786721,79
3,"POLYGON ((514837.673 5038687.087, 514898.502 5...",39.847264,73
4,"POLYGON ((515320.715 5038339.251, 515327.355 5...",39.049800,53
...,...,...,...
6080,"POLYGON ((520945.266 5035272.493, 520829.733 5...",38.206002,3
6081,"POLYGON ((515132.585 5029647.805, 515151.6 502...",39.058524,0
6082,"POLYGON ((515103.596 5029458.179, 515205.059 5...",39.085347,0
6083,"POLYGON ((514410.811 5029162.241, 514667.955 5...",38.893916,0
